In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

drive.mount(
    "/content/drive/",
)
import os

os.chdir(
    "/content/drive/MyDrive/Data science studies/Aprendizado-de-maquina-UFSC/final-project/data"
)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Caminhos dos arquivos
train_path = "./6.turbofan rul/train_FD001.txt"
test_path = "./6.turbofan rul/test_FD001.txt"
rul_path = "./6.turbofan rul/RUL_FD001.txt"

# Nomes das colunas (de acordo com a documentação original do C-MAPSS)
column_names = (
    ["engine_id", "cycle"]
    + [f"op{i}" for i in range(1, 4)]
    + [f"s{i}" for i in range(1, 22)]
)

# Importando os arquivos (espaço em branco como delimitador)
df_train = pd.read_csv(train_path, sep="\s+", header=None, names=column_names)
df_test = pd.read_csv(test_path, sep="\s+", header=None, names=column_names)
df_rul = pd.read_csv(rul_path, sep="\s+", header=None, names=["RUL"])

# Pré-processamento


### Seleção de Sensores

O artigo menciona que apenas 14 dos 21 sensores são usados. Vamos selecionar os mesmos:


In [ ]:
# Sensores selecionados conforme o artigo (índices 2-21, considerando que a indexação começa em 1)
selected_sensors = [
    "s2",
    "s3",
    "s4",
    "s7",
    "s8",
    "s9",
    "s11",
    "s12",
    "s13",
    "s14",
    "s15",
    "s17",
    "s20",
    "s21",
]

# Colunas que vamos manter
features_to_keep = ["engine_id", "cycle"] + selected_sensors + ["op1", "op2", "op3"]

# Filtrando os dataframes
df_train = df_train[features_to_keep]
df_test = df_test[features_to_keep]

### Normalização dos Dados

O artigo usa normalização min-max para o intervalo [-1, 1]:


In [ ]:
def min_max_normalize(df, features):
    """Normaliza as features para o intervalo [-1, 1]"""
    for feature in features:
        if feature in ["engine_id", "cycle"]:
            continue
        min_val = df[feature].min()
        max_val = df[feature].max()
        df[feature] = 2 * (df[feature] - min_val) / (max_val - min_val) - 1
    return df


# Normalizando os dados de treino e teste
features_to_normalize = selected_sensors + ["op1", "op2", "op3"]
df_train = min_max_normalize(df_train, features_to_normalize)
df_test = min_max_normalize(df_test, features_to_normalize)

### Criando os RULs para Treino

O artigo usa um modelo de degradação linear por partes com RUL constante inicial (Re):


In [ ]:
def create_rul_labels(df, Re):
    """Cria os rótulos RUL usando o modelo de degradação linear por partes"""
    # Agrupa por engine_id e encontra o ciclo máximo (falha)
    grouped = df.groupby("engine_id")["cycle"].max().reset_index()
    grouped.columns = ["engine_id", "max_cycle"]

    # Merge para adicionar max_cycle ao dataframe original
    df = df.merge(grouped, on="engine_id", how="left")

    # Calcula o RUL para cada linha
    df["RUL"] = df["max_cycle"] - df["cycle"]

    # Aplica o modelo de degradação linear por partes
    df["RUL"] = df["RUL"].apply(lambda x: Re if x > Re else x)

    return df.drop(columns=["max_cycle"])


# Usando Re=128 conforme sugerido no artigo para FD001
Re = 128
df_train = create_rul_labels(df_train, Re)

### Preparando as Janelas Temporais

O artigo usa janelas temporais com tamanho (nw) e stride (ns):


In [ ]:
def create_time_windows(df, window_size, window_stride, sensor_cols):
    """Cria janelas temporais dos dados dos sensores"""
    sequences = []
    labels = []

    for engine_id in df["engine_id"].unique():
        engine_data = df[df["engine_id"] == engine_id]
        sensor_data = engine_data[sensor_cols].values
        rul_data = engine_data["RUL"].values

        # Cria janelas deslizantes
        for i in range(0, len(engine_data) - window_size + 1, window_stride):
            window = sensor_data[i : i + window_size]
            label = rul_data[i + window_size - 1]  # RUL no último ponto da janela
            sequences.append(window)
            labels.append(label)

    return np.array(sequences), np.array(labels)


# Parâmetros do artigo para FD001: nw=24, ns=1
window_size = 24
window_stride = 1

# Criando as sequências de treino
sensor_cols = selected_sensors + ["op1", "op2", "op3"]
X_train, y_train = create_time_windows(
    df_train, window_size, window_stride, sensor_cols
)

# Reformulando os dados para a MLP (achatando as janelas temporais)
n_samples, n_timesteps, n_features = X_train.shape
X_train = X_train.reshape((n_samples, n_timesteps * n_features))

### Preparando os Dados de Teste

Para os dados de teste, o artigo usa apenas a última janela de cada motor:


def prepare_test_data(df_test, window_size, sensor_cols):
"""Prepara os dados de teste usando a última janela de cada motor"""
X_test = []
engine_ids = []

    for engine_id in df_test['engine_id'].unique():
        engine_data = df_test[df_test['engine_id'] == engine_id]
        sensor_data = engine_data[sensor_cols].values

        # Pega a última janela
        if len(engine_data) >= window_size:
            window = sensor_data[-window_size:]
        else:
            # Preenche com zeros se não houver dados suficientes
            padding = np.zeros((window_size - len(engine_data), n_features))
            window = np.vstack((padding, sensor_data))

        X_test.append(window)
        engine_ids.append(engine_id)

    X_test = np.array(X_test)
    n_samples = X_test.shape[0]
    X_test = X_test.reshape((n_samples, window_size * n_features))

    return X_test, engine_ids

X_test, test_engine_ids = prepare_test_data(df_test, window_size, sensor_cols)


### Carregando os RULs Reais para Teste


In [ ]:
# Carregando os RULs reais
true_rul = df_rul["RUL"].values